# 04 — Prediction Function and API Test

This notebook shows the prediction logic.

In [3]:
from pathlib import Path
import pandas as pd
import joblib


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'data').exists():
            return candidate
    return current


ROOT = find_project_root()
DATA_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'backend' / 'models'

FEATURE_COLUMNS = [
    'mid_rate', 'daily_return', 'return_7d', 'return_14d', 'ma_7', 'ma_14', 'ma_30',
    'ma_gap', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'momentum_7d',
    'momentum_14d', 'spread', 'spread_pct', 'depreciation_days_7d', 'depreciation_days_14d'
]

In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'backend' / 'models'

FEATURE_COLUMNS = [
    'mid_rate', 'daily_return', 'return_7d', 'return_14d', 'ma_7', 'ma_14', 'ma_30',
    'ma_gap', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'momentum_7d',
    'momentum_14d', 'spread', 'spread_pct', 'depreciation_days_7d', 'depreciation_days_14d'
]

def load_latest_feature_row():
    df = pd.read_csv(DATA_DIR / 'exchange_rates_features_and_labels_4year.csv', parse_dates=['date'])
    df = df.dropna(subset=FEATURE_COLUMNS).sort_values('date').reset_index(drop=True)
    return df.iloc[-1]

def risk_guidance(risk):
    if risk == 'High':
        return [
            'Consider paying the supplier earlier if cash flow allows.',
            'Consider splitting the payment to reduce exposure.',
            'Review selling prices or add a margin buffer before confirming quotes.'
        ]
    if risk == 'Medium':
        return [
            'Monitor the USD/RWF rate closely before the payment date.',
            'Review pricing assumptions and prepare a small margin buffer.',
            'Consider partial payment if the invoice is large.'
        ]
    return [
        'Risk is currently low; continue monitoring normally.',
        'The current payment timing appears manageable based on recent signals.',
        'Review again if the payment date is delayed.'
    ]

def pressure_rate(risk, horizon):
    if risk == 'High':
        return 0.012 if horizon == 7 else 0.02
    if risk == 'Medium':
        return 0.006 if horizon == 7 else 0.012
    return 0.0025 if horizon == 7 else 0.005

def predict_fxguard(amount_usd=10000, horizon=7):
    model = joblib.load(MODEL_DIR / f'risk_model_{horizon}d.pkl')
    row = load_latest_feature_row()
    X = pd.DataFrame([row[FEATURE_COLUMNS].astype(float).to_dict()])
    risk = str(model.predict(X)[0])
    confidence = None
    predicted_probability = None
    top_probability_label = None
    probabilities = {}
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X)[0]
        probabilities = {str(cls): round(float(p), 4) for cls, p in zip(model.classes_, probs)}
        top_probability_index = int(max(range(len(probs)), key=lambda i: probs[i]))
        top_probability_label = str(model.classes_[top_probability_index])
        predicted_probability = round(float(probs[top_probability_index]), 4)
        confidence = predicted_probability

    current_rate = float(row['mid_rate'])
    current_cost = amount_usd * current_rate
    extra_cost = current_cost * pressure_rate(risk, horizon)

    return {
        'analysis_date': str(row['date'].date()),
        'pair': 'USD/RWF',
        'amount_usd': amount_usd,
        'horizon_days': horizon,
        'current_rate': round(current_rate, 4),
        'current_cost_rwf': round(current_cost, 2),
        'risk_level': risk,
        'confidence': confidence,
        'confidence_score': confidence,
        'predicted_probability': predicted_probability,
        'top_probability_label': top_probability_label,
        'class_probabilities': probabilities,
        'probability_distribution': probabilities,
        'possible_extra_cost_rwf': round(extra_cost, 2),
        'recommendations': risk_guidance(risk),
        'key_drivers': {
            'daily_return': round(float(row['daily_return']), 6),
            'return_7d': round(float(row['return_7d']), 6),
            'ma_gap': round(float(row['ma_gap']), 6),
            'volatility_7d': round(float(row['volatility_7d']), 6),
            'momentum_7d': round(float(row['momentum_7d']), 6),
            'depreciation_days_7d': int(row['depreciation_days_7d'])
        }
    }

In [4]:
predict_fxguard(amount_usd=10000, horizon=7)

{'analysis_date': '2026-06-25',
 'pair': 'USD/RWF',
 'amount_usd': 10000,
 'horizon_days': 7,
 'current_rate': 1465.445,
 'current_cost_rwf': 14654450.0,
 'risk_level': 'Low',
 'confidence': 0.9228,
 'class_probabilities': {'High': 0.0016, 'Low': 0.9228, 'Medium': 0.0756},
 'possible_extra_cost_rwf': 36636.12,
 'recommendations': ['Risk is currently low; continue monitoring normally.',
  'The current payment timing appears manageable based on recent signals.',
  'Review again if the payment date is delayed.'],
 'key_drivers': {'daily_return': 6.5e-05,
  'return_7d': 0.000314,
  'ma_gap': 0.000503,
  'volatility_7d': 3.1e-05,
  'momentum_7d': 0.000314,
  'depreciation_days_7d': 5}}

In [5]:
predict_fxguard(amount_usd=10000, horizon=14)

{'analysis_date': '2026-06-25',
 'pair': 'USD/RWF',
 'amount_usd': 10000,
 'horizon_days': 14,
 'current_rate': 1465.445,
 'current_cost_rwf': 14654450.0,
 'risk_level': 'Low',
 'confidence': 0.9876,
 'class_probabilities': {'High': 0.0002, 'Low': 0.9876, 'Medium': 0.0123},
 'possible_extra_cost_rwf': 73272.25,
 'recommendations': ['Risk is currently low; continue monitoring normally.',
  'The current payment timing appears manageable based on recent signals.',
  'Review again if the payment date is delayed.'],
 'key_drivers': {'daily_return': 6.5e-05,
  'return_7d': 0.000314,
  'ma_gap': 0.000503,
  'volatility_7d': 3.1e-05,
  'momentum_7d': 0.000314,
  'depreciation_days_7d': 5}}

In [11]:
# Run the backend first: python run_backend.py
import json
from urllib.request import Request, urlopen

payload = {'currency': 'USD', 'amount': 10000, 'horizon': 7, 'margin_percent': 20}
request = Request(
    'http://127.0.0.1:8000/api/predict-risk',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urlopen(request) as response:
    response.status, json.loads(response.read().decode('utf-8'))